# V4.1 — 02: Step 1 Training (Aspect & Opinion Co-Extraction)


## Bootstrap
Mendefinisikan ulang `step_stage`, `require_vars`, dan path modul yang tidak tersimpan di `pipeline_state.pkl`.

In [ ]:
# ============================================================
#  Bootstrap: definisi runtime yang tidak tersimpan di pipeline_state.pkl
#  Dijalankan otomatis sebelum state recovery di setiap notebook serial.
# ============================================================
import os, sys, time, json, re, pickle, shutil, glob, importlib, warnings
from datetime import datetime

# --- step_stage & require_vars (dari sel 6 V4.1) ---
class step_stage:
    def __init__(self, title, total_steps=None):
        self.title = title
        self.total = total_steps
        self.n = 0
        self.t0 = None
    def __enter__(self):
        self.t0 = time.time()
        print("=" * 78)
        print(f"\u25b6\ufe0f  {self.title}")
        print("=" * 78, flush=True)
        return self
    def step(self, msg):
        self.n += 1
        tag = f"{self.n}/{self.total}" if self.total else str(self.n)
        print(f"   [{tag}] {time.time() - self.t0:6.1f}s  {msg}", flush=True)
    def note(self, msg):
        print(f"        {msg}", flush=True)
    def __exit__(self, exc_type, exc, tb):
        dur = time.time() - self.t0
        if exc_type is None:
            print(f"\u2705 {self.title} \u2014 selesai dalam {dur:.1f}s\n", flush=True)
        else:
            print(f"\u274c {self.title} \u2014 gagal setelah {dur:.1f}s: {exc}\n", flush=True)
        return False

def require_vars(*names):
    missing = [n for n in names if n not in globals()]
    if missing:
        raise RuntimeError(
            f"Variabel {missing} belum ada di memori. Jalankan sel pemulihan state "
            f"(6b/6c) atau notebook 01_setup lebih dulu.")

def write_stage_progress(path, **fields):
    d = {"saved_at": datetime.now().isoformat()}
    d.update(fields)
    if os.path.exists(path):
        try:
            with open(path, "r", encoding="utf-8") as f:
                old = json.load(f)
            if isinstance(old, dict):
                d["previous"] = old
        except Exception:
            pass
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(d, f, indent=2, ensure_ascii=False)

def _prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f1

# --- patch_eval_metrics_counts (ringkas; full patch dijalankan ulang di sel 5a/8a) ---
# Pastikan nama fungsi tersedia agar require_vars tidak gagal bila sel import
# upstream belum berjalan. Patch sebenarnya dilakukan di sel step1/step2 init.
def patch_eval_metrics_counts():
    try:
        import eval_metrics as _em
    except ImportError:
        return "eval_metrics belum di-import (akan dipatch di sel init)"
    return "eval_metrics siap"

def history_display_frame(history, epochs_col="epoch"):
    import pandas as pd
    if not history:
        return pd.DataFrame()
    return pd.DataFrame(history)

def metrics_display_frame(res):
    import pandas as pd
    if not res:
        return pd.DataFrame()
    rows = [{"Metric": k, "Value": v} for k, v in res.items()]
    return pd.DataFrame(rows)

def best_epoch_row(history, f1_key="micro-F1"):
    if not history:
        return None, 0.0, None
    best = max(history, key=lambda r: float(r.get(f1_key, 0.0)))
    return best, float(best.get(f1_key, 0.0)), int(best.get("epoch", 0))

def unpack_model_output(out):
    losses, logits = out
    loss = losses[0] if isinstance(losses, (list, tuple)) else losses
    return loss, logits

# --- Path setup (dari sel 8 V4.1, versi ringkas) ---
IS_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if "base_project_dir" not in globals() or not globals().get("base_project_dir"):
    if os.path.exists("/content/drive/MyDrive/ACOS"):
        base_project_dir = "/content/drive/MyDrive/ACOS"
    elif os.path.exists("/content/ACOS"):
        base_project_dir = "/content/ACOS"
    else:
        base_project_dir = os.path.abspath(".")
    extract_dir = os.path.join(base_project_dir, "Extract-Classify-ACOS")
    data_root = os.path.join(base_project_dir, "data")

# sys.path: upstream ACOS + acos_id
for _p in [extract_dir, os.path.join(extract_dir, "absa5")]:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

# --- acos_id reimport ---
def _cari_indo_root():
    for _d in ["/content/drive/MyDrive/ACOS-IndoBERT",
               "/content/drive/MyDrive/ACOS/ACOS-IndoBERT",
               "/content/drive/MyDrive/ACOS-ASLI/ACOS-IndoBERT",
               os.path.join(base_project_dir, "ACOS-IndoBERT"),
               os.path.abspath("ACOS-IndoBERT"),
               os.path.abspath(os.path.join("..", "ACOS-IndoBERT"))]:
        if os.path.isdir(os.path.join(_d, "acos_id")):
            return _d
    return None

indo_root = globals().get("indo_root") or _cari_indo_root()
if indo_root and indo_root not in sys.path:
    sys.path.insert(0, indo_root)
if indo_root:
    try:
        acos_id = importlib.import_module("acos_id")
        acos_taxonomy = importlib.import_module("acos_id.taxonomy")
        acos_selftest = importlib.import_module("acos_id.selftest")
        acos_ckpt = importlib.import_module("acos_id.checkpoint")
        acos_eda = importlib.import_module("acos_id.eda")
        acos_upstream = importlib.import_module("acos_id.upstream")
    except ModuleNotFoundError:
        pass  # akan dipatch di sel init masing-masing step

# _backbone_dirname untuk recovery state
BACKBONE_DIRNAME = {
    "indobert": "indobert_base_p1",
    "indobert-large": "indobert_large_p1",
    "bert-en": "bert_base_uncased",
}
def _backbone_dirname(backbone=None):
    key = backbone or globals().get("BACKBONE") or "bert-en"
    return BACKBONE_DIRNAME.get(key, str(key).replace("-", "_"))

print(f"\u26a1 Bootstrap siap | base_project_dir={base_project_dir} | indo_root={indo_root}")


## State Recovery
Memuat `pipeline_state.pkl` dari sesi sebelumnya. Jika belum ada, jalankan notebook `01_setup` lebih dulu.

### 6b. Smart State Recovery (Gunakan Sel Ini Jika Kernel Reconnect / Restart)

In [ ]:
# Sel Pemulihan Cerdas: Otomatis mendeteksi sesi aktif terakhir.
# Memulihkan BUKAN hanya config/path, tapi juga seluruh artefak runtime yang tersimpan.
def auto_find_latest_state(search_bases, domain="rest16"):
    """Mencari berkas state pipeline_state.pkl dengan validasi domain."""
    if isinstance(search_bases, str):
        search_bases = [search_bases]

    # 1. Cek pointer langsung
    for sb in search_bases:
        if not sb or not os.path.isdir(sb):
            continue
        pointer = os.path.join(sb, f"latest_pipeline_state_{domain}.pkl")
        if os.path.exists(pointer):
            return pointer

    # 2. Cari mendalam
    candidates = []
    for sb in search_bases:
        if not sb or not os.path.exists(sb):
            continue
        for root, dirs, files in os.walk(sb):
            if "pipeline_state.pkl" in files:
                p = os.path.join(root, "pipeline_state.pkl")
                norm = p.replace(os.sep, "/")
                if domain and f"/{domain}_" not in norm and f"_{domain}/" not in norm and f"/{domain}/" not in norm:
                    try:
                        with open(p, "rb") as f:
                            s = pickle.load(f)
                        if s.get("DOMAIN") != domain:
                            continue
                    except Exception:
                        continue
                candidates.append((os.path.getmtime(p), p))
    if candidates:
        candidates.sort(reverse=True)
        return candidates[0][1]
    return None


## 5. Step 1: Aspect & Opinion Co-Extraction (BERT-CRF)
Tahap ini dipecah menjadi enam sel (5a-5f) agar setiap bagian punya progres dan durasi sendiri,
sehingga kegagalan atau kelambatan bisa dilacak ke satu tahap saja. Jalankan berurutan.

| Sel | Isi | Aman diulang |
|---|---|---|
| 5a | Import, tokenizer, label map, resolusi path checkpoint | ya |
| 5b | Deteksi cache (sesi aktif + sesi lama), keputusan latih/lewati | ya |
| 5c | Data evaluasi + ground truth (`eval_gold_1`) | ya |
| 5d | Instansiasi model, data training, optimizer | ya (mengalokasi ulang VRAM) |
| 5e | Loop training per epoch + checkpoint terbaik | tidak (melatih ulang) |
| 5f | Plot, tabel laporan, manifest, simpan state | ya |

Sel 5c-5e melewati dirinya sendiri secara otomatis saat `STEP1_SKIP_TRAINING` bernilai `True`.
Set `FORCE_RETRAIN_STEP1 = True` di sel 5a untuk memaksa training ulang.

In [ ]:
from modeling import BertForQuadABSA
from bert_utils.tokenization import BertTokenizer
from bert_utils.optimization import BertAdam
from run_classifier_dataset_utils import processors, output_modes
from eval_metrics import pred_eval
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from tqdm.auto import tqdm
import time


In [ ]:
require_vars("step_stage", "session_dirs", "bert_cache_dir", "DOMAIN")

from modeling import BertForQuadABSA
from bert_utils.tokenization import BertTokenizer
from bert_utils.optimization import BertAdam
from run_classifier_dataset_utils import processors, output_modes
from eval_metrics import pred_eval
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from tqdm.auto import tqdm

# Toggle Melatih Ulang (Set True jika ingin melatih ulang dari awal)
FORCE_RETRAIN_STEP1 = False

with step_stage("5a. Inisialisasi Step 1: tokenizer, patch metrik, taksonomi ID, label, path", 7) as st:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            _vram_txt = f"VRAM bebas {torch.cuda.mem_get_info()[0] / 1024 ** 3:.2f} GB"
        except Exception:  # mem_get_info tidak ada di torch lama
            _vram_txt = (f"VRAM total "
                         f"{torch.cuda.get_device_properties(0).total_memory / 1024 ** 3:.2f} GB")
        st.step(f"GPU siap: {torch.cuda.get_device_name(0)} | {_vram_txt}")
    else:
        st.step("Mode CPU aktif (CUDA tidak tersedia) — training akan jauh lebih lambat")

    tokenizer = BertTokenizer.from_pretrained(bert_cache_dir, do_lower_case=True)
    st.step(f"Tokenizer dimuat: {len(tokenizer.vocab):,} entri vocab dari {bert_cache_dir}")

    _em1 = patch_eval_metrics_counts()
    st.step("eval_metrics dipatch: measureQuad & measureQuad_imp kini mengembalikan "
            "tp/fp/fn (+ agregat semua slot difficulty)")

    # Domain Indonesia: get_labels() upstream hanya mengenal 'rest*' dan 'laptop';
    # domain lain membiarkan daftar kategori None lalu meledak di `for cate in l`.
    # Patch runtime menambah cabangnya tanpa menyentuh berkas upstream, sehingga
    # jalur Inggris tetap utuh sebagai kontrol.
    _lab_patch = acos_taxonomy.patch_processor_labels(processors)
    st.step(f"Taksonomi Indonesia: {_lab_patch}")

    processor_step1 = processors["quad"]()
    label_list_step1 = processor_step1.get_labels(DOMAIN)
    num_labels_step1 = len(label_list_step1[1])
    label_map_seq = {label: i for i, label in enumerate(label_list_step1[1])}
    st.step(f"Label sekuens ({num_labels_step1}): {label_list_step1[1]}")

    step1_ckpt = session_dirs["step1_checkpoint"]
    step1_bin = os.path.join(step1_ckpt, "pytorch_model.bin")
    step1_csv = os.path.join(session_dirs["csv"], "step1_training_history.csv")
    pred_file = os.path.join(session_dirs["logs"], "pred4pipeline.txt")
    step1_progress_json = os.path.join(session_dirs["logs"], "step1_progress.json")
    os.makedirs(step1_ckpt, exist_ok=True)
    st.step(f"Checkpoint  : {step1_ckpt}")
    st.step(f"Prediksi    : {pred_file} | FORCE_RETRAIN_STEP1={FORCE_RETRAIN_STEP1} | "
            f"epoch target={NUM_EPOCHS}")

### 5b. Deteksi Cache Step 1
Memeriksa artefak sesi aktif, lalu menarik checkpoint dari sesi lama bila perlu.
Hasilnya adalah `STEP1_SKIP_TRAINING`, satu-satunya penentu apakah sel 5c-5e berjalan.

In [ ]:
require_vars("step_stage", "step1_bin", "pred_file", "step1_csv", "FORCE_RETRAIN_STEP1")

with step_stage("5b. Deteksi cache Step 1 (sesi aktif lalu sesi lama)", 4) as st:
    step1_already_done = os.path.exists(step1_bin) and os.path.exists(pred_file)
    st.step("Sesi aktif — model: {} | pred4pipeline: {}".format(
        f"{os.path.getsize(step1_bin) / 1024 ** 2:.1f} MB" if os.path.exists(step1_bin) else "belum ada",
        f"{sum(1 for _ in open(pred_file, encoding='utf-8'))} baris" if os.path.exists(pred_file) else "belum ada"))

    if step1_already_done:
        st.step("Pencarian sesi lama dilewati (artefak sesi aktif sudah lengkap)")
    else:
        found_bin = auto_find_file("pytorch_model.bin", must_contain="step1_best", search_roots=[
            results_base if 'results_base' in globals() else "",
            "/content/drive/MyDrive/ACOS/Output/results",
            os.path.join(base_project_dir, "Output", "results"),
        ])
        if found_bin and "step1_best" in found_bin:
            src_dir = os.path.dirname(found_bin)
            st.step(f"Checkpoint sesi sebelumnya ditemukan: {src_dir}")
            for fn in ["pytorch_model.bin", "config.json", "vocab.txt"]:
                fp = os.path.join(src_dir, fn)
                if os.path.exists(fp):
                    shutil.copy(fp, os.path.join(step1_ckpt, fn))
                    st.note(f"↪ {fn} ({os.path.getsize(fp) / 1024 ** 2:.1f} MB) disalin ke sesi aktif")
            found_pred = auto_find_file("pred4pipeline.txt")
            if found_pred:
                shutil.copy(found_pred, pred_file)
                st.note(f"↪ pred4pipeline.txt disalin dari {found_pred}")
            found_csv = auto_find_file("step1_training_history.csv")
            if found_csv:
                shutil.copy(found_csv, step1_csv)
                st.note(f"↪ step1_training_history.csv disalin dari {found_csv}")
            step1_already_done = os.path.exists(step1_bin) and os.path.exists(pred_file)
        else:
            st.step("Tidak ada checkpoint step1_best di sesi mana pun")

    STEP1_SKIP_TRAINING = (not FORCE_RETRAIN_STEP1) and step1_already_done
    st.step("Keputusan: " + ("CACHE HIT → sel 5c-5e dilewati"
                             if STEP1_SKIP_TRAINING else
                             f"TRAINING dijalankan ({NUM_EPOCHS} epoch)"))

    if STEP1_SKIP_TRAINING:
        print(f"⏩ [CACHE HIT] Model Step 1 : {step1_ckpt}")
        print(f"⏩ [CACHE HIT] Prediksi     : {pred_file}")
        if os.path.exists(step1_csv):
            df_s1_saved = pd.read_csv(step1_csv)
            step1_history = df_s1_saved.to_dict('records')
            _row_c1, best_step1_f1, best1_epoch = best_epoch_row(step1_history)
            best1_epoch = best1_epoch or NUM_EPOCHS
            st.step(f"Riwayat tersimpan: {len(df_s1_saved)} epoch, terbaik epoch {best1_epoch}")
            if len(df_s1_saved) < NUM_EPOCHS:
                st.note(f"⚠️ Riwayat hanya {len(df_s1_saved)}/{NUM_EPOCHS} epoch — checkpoint ini "
                        f"berasal dari run yang terhenti. Set FORCE_RETRAIN_STEP1=True bila "
                        f"ingin melatih penuh.")
        else:
            step1_history = []
            best_step1_f1 = 0.0
            best1_epoch = NUM_EPOCHS
            st.step("Riwayat CSV tidak ada — metrik per epoch tidak bisa dilaporkan")
# ── Resume Epoch: cek apakah training Step 1 terhenti di tengah jalan ────────
step1_resume_json = os.path.join(session_dirs["logs"], "step1_resume.json")
STEP1_RESUME_EPOCH = 0  # epoch terakhir yang selesai (0 = belum ada / baru)

if (not STEP1_SKIP_TRAINING) and (not FORCE_RETRAIN_STEP1):
    if not os.path.exists(step1_resume_json):
        _found_resume = auto_find_file("step1_resume.json", search_roots=[
            results_base if 'results_base' in globals() else "",
            "/content/drive/MyDrive/ACOS/Output/results",
            os.path.join(base_project_dir, "Output", "results"),
        ])
        if _found_resume and os.path.exists(_found_resume):
            try:
                _rj_prev = json.load(open(_found_resume, encoding="utf-8"))
                _last_prev = int(_rj_prev.get("last_completed_epoch", 0))
                _prev_session_root = os.path.dirname(os.path.dirname(_found_resume))
                _prev_epoch_dir = os.path.join(_prev_session_root, "checkpoints", f"step1_epoch_{_last_prev}")
                if os.path.isdir(_prev_epoch_dir):
                    _tgt_epoch_dir = os.path.join(session_dirs["checkpoints"], f"step1_epoch_{_last_prev}")
                    os.makedirs(_tgt_epoch_dir, exist_ok=True)
                    for _f in os.listdir(_prev_epoch_dir):
                        shutil.copy(os.path.join(_prev_epoch_dir, _f), os.path.join(_tgt_epoch_dir, _f))
                    shutil.copy(_found_resume, step1_resume_json)
                    print(f"↪ [RESUME] Menyalin resume checkpoint dari sesi lama: epoch {_last_prev}")
            except Exception as _e_copy:
                print(f"⚠️ Gagal menyalin resume dari sesi lama: {_e_copy}")

    if os.path.exists(step1_resume_json):
        try:
            _rj = json.load(open(step1_resume_json, encoding="utf-8"))
            _last = int(_rj.get("last_completed_epoch", 0))
            _total = int(_rj.get("total_epochs", NUM_EPOCHS))
            _epoch_ckpt_check = os.path.join(
                session_dirs["checkpoints"], f"step1_epoch_{_last}")
            _epoch_bin_check  = os.path.join(_epoch_ckpt_check, "pytorch_model.bin")
            if _total != NUM_EPOCHS:
                print(f"⚠️  [RESUME] NUM_EPOCHS berubah ({_total}→{NUM_EPOCHS}). "
                      f"Resume diabaikan — training dari awal.")
            elif _last > 0 and _last < NUM_EPOCHS and os.path.exists(_epoch_bin_check):
                STEP1_RESUME_EPOCH = _last
                print(f"♻️  [RESUME] Step 1 terhenti di epoch {_last}/{NUM_EPOCHS}. "
                      f"Akan dilanjutkan dari epoch {_last + 1}.")
            elif _last >= NUM_EPOCHS:
                print(f"✅ [RESUME] step1_resume.json menunjukkan training sudah "
                      f"selesai ({_last}/{NUM_EPOCHS} epoch).")
        except Exception as _re:
            print(f"⚠️  Tidak bisa membaca step1_resume.json: {_re}. Training dari awal.")

elif STEP1_SKIP_TRAINING:
    if os.path.exists(step1_resume_json):
        try:
            _rj_info = json.load(open(step1_resume_json, encoding="utf-8"))
            _ep_info = _rj_info.get("last_completed_epoch", "?")
            print(f"ℹ️  step1_resume.json ada (epoch {_ep_info}/{NUM_EPOCHS}) "
                  f"— diabaikan karena cache hit sudah lengkap.")
        except Exception:
            pass

if STEP1_RESUME_EPOCH > 0:
    STEP1_SKIP_TRAINING = False  # paksa jalankan training (lanjut dari epoch berikutnya)
    print(f"   → STEP1_SKIP_TRAINING di-override: training akan lanjut dari "
          f"epoch {STEP1_RESUME_EPOCH + 1}.")

### 5c. Data Evaluasi & Ground Truth
Membangun `eval_loader_1` dan `eval_gold_1`. `eval_gold_1` memakai `aspect_input_ids` hasil
fitur, bukan id token mentah; memakai id mentah menggeser teks `pred4pipeline.txt` satu token
dan mengosongkan kalimat satu kata.

In [ ]:
require_vars("step_stage", "STEP1_SKIP_TRAINING", "label_map_seq", "tokenizer")

if STEP1_SKIP_TRAINING:
    print("⏩ 5c dilewati — memakai artefak Step 1 dari cache (lihat sel 5b).")
else:
    with step_stage("5c. Data evaluasi test + ground truth", 5) as st:
        eval_examples_1 = processor_step1.get_dev_examples(tokenized_base, DOMAIN)
        st.step(f"{len(eval_examples_1):,} contoh test dibaca dari "
                f"tokenized_data/{DOMAIN}_test_quad_bert.tsv")

        eval_features_1 = features_step1(eval_examples_1, label_list_step1, MAX_SEQ_LENGTH,
                                         tokenizer, output_modes["quad"], "quad")
        st.step(f"{len(eval_features_1):,} fitur dibentuk (max_seq_length={MAX_SEQ_LENGTH})")

        ev_ids = torch.tensor([f.aspect_input_ids for f in eval_features_1], dtype=torch.long)
        ev_mask = torch.tensor([f.aspect_input_mask for f in eval_features_1], dtype=torch.long)
        ev_seg = torch.tensor([f.aspect_segment_ids for f in eval_features_1], dtype=torch.long)
        ev_lbl = torch.tensor([f.aspect_ids for f in eval_features_1], dtype=torch.long)
        ev_imp_a = torch.tensor([f.exist_imp_aspect for f in eval_features_1], dtype=torch.long)
        ev_imp_o = torch.tensor([f.exist_imp_opinion for f in eval_features_1], dtype=torch.long)
        ev_len = torch.tensor([f.tokens_len for f in eval_features_1], dtype=torch.long)
        eval_data_1 = TensorDataset(ev_len, ev_ids, ev_mask, ev_lbl, ev_seg, ev_imp_a, ev_imp_o)

        pin_mem = torch.cuda.is_available()
        num_work = 0 if sys.platform.startswith('win') else 2
        eval_loader_1 = DataLoader(
            eval_data_1, sampler=SequentialSampler(eval_data_1),
            batch_size=16, pin_memory=pin_mem, num_workers=num_work
        )
        st.step(f"eval_loader_1 siap: {len(eval_loader_1)} batch × 16 "
                f"(pin_memory={pin_mem}, workers={num_work})")

        # Ground truth: input_text wajib memakai aspect_input_ids hasil feature
        # ([CLS] .. [CLS] + zero-pad). pred_eval menulis pred4pipeline.txt dari
        # ids_to_token[1:tokens_len-1]; id token mentah menggeser teks satu token.
        gold_tsv = os.path.join(tokenized_base, "tokenized_data", f"{DOMAIN}_test_quad_bert.tsv")
        with open(gold_tsv, "r", encoding="utf-8") as f:
            gold_lines = f.readlines()

        eval_gold_labels = []
        n_imp_a = n_imp_o = 0
        for line in tqdm(gold_lines, desc="   Parsing gold quad", unit="baris", leave=False):
            line = line.strip().split("\t")
            aspect_labels = [label_map_seq['O'] for _ in range(MAX_SEQ_LENGTH)]
            cur_imp_a, cur_imp_o = 0, 0
            for quad in line[1:]:
                cur_aspect, cur_opinion = quad.split(' ')[0], quad.split(' ')[-1]
                a_st, a_ed = int(cur_aspect.split(',')[0]), int(cur_aspect.split(',')[1])
                if a_ed != -1:
                    aspect_labels[a_st] = label_map_seq['B-A']
                    for i in range(a_st + 1, a_ed):
                        aspect_labels[i] = label_map_seq['I-A']
                else:
                    cur_imp_a = 1
                o_st, o_ed = int(cur_opinion.split(',')[0]), int(cur_opinion.split(',')[1])
                if o_ed != -1:
                    aspect_labels[o_st] = label_map_seq['B-O']
                    for i in range(o_st + 1, o_ed):
                        aspect_labels[i] = label_map_seq['I-O']
                else:
                    cur_imp_o = 1
            eval_gold_labels += [aspect_labels, cur_imp_a, cur_imp_o]
            n_imp_a += cur_imp_a
            n_imp_o += cur_imp_o
        st.step(f"Gold terbentuk untuk {len(gold_lines):,} kalimat "
                f"(implicit aspect: {n_imp_a}, implicit opinion: {n_imp_o})")

        eval_gold_1 = [ev_ids.numpy().tolist(), eval_gold_labels]
        assert len(eval_gold_labels) == 3 * len(eval_features_1), (
            f"Gold ({len(eval_gold_labels) // 3} kalimat) tidak sejajar dengan fitur "
            f"({len(eval_features_1)}). pred_eval akan salah memetakan prediksi.")
        st.step("eval_gold_1 sejajar dengan eval_loader_1 — siap dipakai pred_eval")

### 5d. Model, Data Training & Optimizer
Mengalokasikan VRAM untuk `BertForQuadABSA` dan menyiapkan `BertAdam`.
Jalankan ulang sel ini bila ingin mereset bobot ke pretrained sebelum melatih lagi.

In [ ]:
require_vars("step_stage", "STEP1_SKIP_TRAINING")

if STEP1_SKIP_TRAINING:
    print("⏩ 5d dilewati — model dan optimizer tidak diperlukan saat cache hit.")
else:
    require_vars("eval_loader_1", "eval_gold_1")
    with step_stage("5d. Model BERT-CRF, data training, optimizer", 5) as st:
        model_step1 = BertForQuadABSA.from_pretrained(
            bert_cache_dir, num_labels=num_labels_step1).to(device)
        _n_par = sum(p.numel() for p in model_step1.parameters())
        _vram = torch.cuda.memory_allocated(device) / 1024 ** 2 if torch.cuda.is_available() else 0.0
        st.step(f"Model dimuat ke {device}: {_n_par / 1e6:.1f} M parameter, "
                f"VRAM terpakai {_vram:.0f} MB")

        train_examples_1 = processor_step1.get_train_examples(tokenized_base, DOMAIN)
        st.step(f"{len(train_examples_1):,} contoh training dibaca")

        train_features_1 = features_step1(train_examples_1, label_list_step1, MAX_SEQ_LENGTH,
                                          tokenizer, output_modes["quad"], "quad")
        tr_data_1 = TensorDataset(
            torch.tensor([f.tokens_len for f in train_features_1], dtype=torch.long),
            torch.tensor([f.aspect_input_ids for f in train_features_1], dtype=torch.long),
            torch.tensor([f.aspect_input_mask for f in train_features_1], dtype=torch.long),
            torch.tensor([f.aspect_ids for f in train_features_1], dtype=torch.long),
            torch.tensor([f.aspect_segment_ids for f in train_features_1], dtype=torch.long),
            torch.tensor([f.exist_imp_aspect for f in train_features_1], dtype=torch.long),
            torch.tensor([f.exist_imp_opinion for f in train_features_1], dtype=torch.long)
        )
        train_loader_1 = DataLoader(
            tr_data_1, sampler=RandomSampler(tr_data_1),
            batch_size=STEP1_BATCH_SIZE, pin_memory=pin_mem, num_workers=num_work
        )
        st.step(f"train_loader_1 siap: {len(train_loader_1)} batch × {STEP1_BATCH_SIZE}")

        num_train_steps_1 = len(train_loader_1) * NUM_EPOCHS
        param_opt = list(model_step1.named_parameters())
        no_decay = ['bias', 'LayerNorm.bias', 'LayerNorm.weight']
        opt_grouped = [
            {'params': [p for n, p in param_opt if not any(nd in n for nd in no_decay)], 'weight_decay': 0.01},
            {'params': [p for n, p in param_opt if any(nd in n for nd in no_decay)], 'weight_decay': 0.0}
        ]
        optimizer_1 = BertAdam(opt_grouped, lr=STEP1_LR, warmup=0.1, t_total=num_train_steps_1)
        st.step(f"BertAdam siap: lr={STEP1_LR}, warmup=0.1, t_total={num_train_steps_1:,} step")

        class ArgsH:
            def __init__(self):
                self.output_dir = session_dirs["logs"]
                self.max_seq_length = MAX_SEQ_LENGTH

        args_h = ArgsH()
        import logging
        logger = logging.getLogger("Step1")
        st.step(f"args_h siap — pred_eval akan menulis pred4pipeline.txt ke {args_h.output_dir}")

### 5d2. Gate 1 — Bobot Encoder Benar-Benar Termuat

Gate paling penting di seluruh notebook. Sel ini membandingkan tiga tensor
encoder di model yang **sudah dimuat** dengan tensor yang sama di checkpoint di
disk, memakai `torch.equal`, bukan sekadar memeriksa nama key.

Tensor yang diperiksa: embedding kata, `layer.0` query, dan `layer.11` output —
layer pertama dan terakhir supaya kegagalan sebagian juga tertangkap.

Kalau gate ini merah, semua yang di bawahnya percuma: encoder terinisialisasi
acak dan angka F1 yang keluar mengukur kemampuan head belajar dari
representasi acak. Karena logging `missing_keys` upstream di-comment out, tidak
ada gejala lain yang muncul.

In [ ]:
require_vars("step_stage", "acos_ckpt", "bert_cache_dir")

with step_stage("5d2. Gate 1: bobot IndoBERT benar-benar termuat", 4) as st:
    if STEP1_SKIP_TRAINING:
        st.step("Step 1 memakai cache — model belum dimuat, gate dilewati")
        st.note("Gate 1 hanya bermakna pada model yang baru di-from_pretrained. "
                "Set FORCE_RETRAIN_STEP1=True bila ingin memverifikasi ulang.")
        gate1_report = {"dilewati": True}
    elif not acos_taxonomy.is_id_domain(DOMAIN):
        st.step(f"DOMAIN='{DOMAIN}' — checkpoint Inggris sudah berprefiks bert., "
                f"gate tetap dijalankan sebagai kontrol")
        gate1_report = acos_ckpt.gate_weights_loaded(model_step1, bert_cache_dir)
    else:
        require_vars("model_step1")
        gate1_report = acos_ckpt.gate_weights_loaded(model_step1, bert_cache_dir)

    if not gate1_report.get("dilewati"):
        for _name, _r in gate1_report["tensor"].items():
            _short = _name.replace("bert.encoder.layer.", "layer").replace(
                "bert.embeddings.", "emb.")
            if _r["status"] == "LULUS":
                st.step(f"✅ {_short} — identik (mean {_r['mean_model']:+.6f})")
            else:
                st.step(f"❌ {_short} — {_r.get('alasan', 'tidak cocok')}")
        st.step(f"Key bert.* di model={gate1_report['n_key_bert_model']}, "
                f"di checkpoint={gate1_report['n_key_bert_checkpoint']}, "
                f"tanpa padanan={gate1_report['n_key_model_tanpa_padanan']}")

        df_gate1 = pd.DataFrame([
            {"Tensor": k.replace("bert.", ""), "Status": v["status"],
             "Bentuk": str(v.get("bentuk", "")),
             "Mean_Checkpoint": v.get("mean_checkpoint"),
             "Mean_Model": v.get("mean_model")}
            for k, v in gate1_report["tensor"].items()])
        export_step_table(df_gate1, name="master_00c_gate1_bobot_encoder",
                          csv_dir=csv_dir, md_dir=md_dir,
                          title="Gate 1 — Verifikasi Bobot Encoder")
        rep.table(df_gate1, caption="Gate 1: bobot encoder vs checkpoint")

        with open(os.path.join(session_dirs["logs"], "gate1_weights.json"),
                  "w", encoding="utf-8") as jf:
            json.dump(gate1_report, jf, indent=2, ensure_ascii=False, default=str)

        if not gate1_report["ok"]:
            raise RuntimeError(
                "GATE 1 GAGAL: bobot encoder di model tidak sama dengan checkpoint. "
                "Encoder kemungkinan terinisialisasi acak (modeling.py:745 memakai "
                "start_prefix='' karena kelas punya self.bert, sehingga key tanpa "
                "prefiks 'bert.' tidak pernah termuat). Jalankan ulang sel 4c dengan "
                "acos_ckpt.prepare_backbone(BACKBONE, bert_cache_dir, force_rekey=True).")
        st.step("Gate 1 LULUS — fine-tuning berjalan di atas bobot IndoBERT terlatih")

### 5e. Loop Training Step 1
Progres berjalan di tiga tingkat: bar epoch (ETA total), bar batch (loss berjalan), dan
ringkasan satu baris per epoch. Setiap epoch juga menulis `csv/step1_training_history.csv`,
`logs/step1_progress.json`, dan `session_manifest.json`, jadi progres tetap terbaca dari
berkas kalau tab Colab tertutup.

In [ ]:
require_vars("step_stage", "STEP1_SKIP_TRAINING")

if STEP1_SKIP_TRAINING:
    print("⏩ 5e dilewati — training Step 1 tidak dijalankan (cache hit).")
    print(f"   Micro-F1 terbaik tersimpan: {best_step1_f1 * 100:.2f}% (epoch {best1_epoch})")
else:
    require_vars("model_step1", "optimizer_1", "train_loader_1", "eval_loader_1")
    with step_stage(f"5e. Training Step 1 BERT-CRF — {NUM_EPOCHS} epoch pada {device}",
                    NUM_EPOCHS) as st:
        # ── Resume State ─────────────────────────────────────────────────────
        step1_resume_json = os.path.join(session_dirs["logs"], "step1_resume.json")
        start_epoch    = 1
        best_step1_f1  = 0.0
        best1_epoch    = 1
        step1_history  = []
        epochs_since_best_1 = 0  # Counter untuk early stopping
        early_stopped_step1 = False  # Flag apakah training berhenti karena early stopping

        _resume_ep = globals().get("STEP1_RESUME_EPOCH", 0)
        if _resume_ep > 0 and not FORCE_RETRAIN_STEP1:
            _epoch_ckpt_r = os.path.join(
                session_dirs["checkpoints"], f"step1_epoch_{_resume_ep}")
            _model_path_r = os.path.join(_epoch_ckpt_r, "pytorch_model.bin")
            _opt_path_r   = os.path.join(_epoch_ckpt_r, "optimizer.pt")
            try:
                model_step1.load_state_dict(
                    torch.load(_model_path_r, map_location=device))
                st.note(f"✅ Bobot model direstorasi dari epoch {_resume_ep}")
                if os.path.exists(_opt_path_r):
                    optimizer_1.load_state_dict(
                        torch.load(_opt_path_r, map_location="cpu"))
                    st.note(f"✅ Optimizer state direstorasi dari epoch {_resume_ep}")
                else:
                    st.note(f"⚠️  optimizer.pt tidak ada — optimizer mulai baru "
                            f"(bobot model tetap dari epoch {_resume_ep})")
                _rj_r = json.load(open(step1_resume_json, encoding="utf-8"))
                step1_history = _rj_r.get("history", [])
                best_step1_f1 = float(_rj_r.get("best_micro_f1", 0.0))
                best1_epoch   = int(_rj_r.get("best_epoch", 1))
                start_epoch   = _resume_ep + 1
                st.step(f"♻️  Resume dari epoch {_resume_ep} → mulai epoch {start_epoch} "
                        f"| best F1 sejauh ini: {best_step1_f1 * 100:.2f}% "
                        f"(epoch {best1_epoch})")
            except Exception as _load_err:
                st.note(f"❌ Gagal load resume checkpoint: {_load_err}. "
                        f"Training dimulai dari awal (epoch 1).")
                start_epoch   = 1
                best_step1_f1 = 0.0
                best1_epoch   = 1
                step1_history = []
        else:
            st.step("Memulai training baru dari epoch 1")
        # ─────────────────────────────────────────────────────────────────────

        _max_run = globals().get("MAX_EPOCHS_THIS_RUN", 0)
        _run_until = (start_epoch + _max_run - 1) if _max_run else NUM_EPOCHS
        _run_until = min(_run_until, NUM_EPOCHS)
        _use_amp = globals().get("USE_AMP", True) and torch.cuda.is_available()
        scaler1 = torch.cuda.amp.GradScaler(enabled=_use_amp)
        epoch_bar = tqdm(range(start_epoch, _run_until + 1), desc="Step 1 epoch",
                         unit="epoch", initial=start_epoch - 1, total=_run_until)
        for epoch in epoch_bar:
            model_step1.train()
            t_loss = 0.0
            batch_bar = tqdm(train_loader_1, desc=f"  epoch {epoch}/{NUM_EPOCHS}",
                             unit="batch", leave=False)
            for step, batch in enumerate(batch_bar, 1):
                batch = tuple(t.to(device) for t in batch)
                _len, _ids, _mask, _lbls, _seg, _imp_a, _imp_o = batch
                with torch.cuda.amp.autocast(enabled=_use_amp):
                    out1 = model_step1(aspect_input_ids=_ids, aspect_labels=_lbls,
                                       aspect_token_type_ids=_seg, aspect_attention_mask=_mask,
                                       exist_imp_aspect=_imp_a, exist_imp_opinion=_imp_o)
                loss, _ = unpack_model_output(out1)
                scaler1.scale(loss).backward()
                scaler1.step(optimizer_1)
                scaler1.update()
                optimizer_1.zero_grad(set_to_none=True)
                t_loss += loss.item()
                if step % 10 == 0 or step == len(train_loader_1):
                    batch_bar.set_postfix(loss=f"{t_loss / step:.4f}")
            batch_bar.close()

            avg_loss = t_loss / len(train_loader_1)
            model_step1.eval()
            print(f"   Epoch {epoch:02d}: evaluasi test set ({len(eval_loader_1)} batch)...",
                  flush=True)
            val_res = pred_eval(epoch, args_h, logger, tokenizer, model_step1, eval_loader_1,
                                eval_gold_1, label_list_step1, device, "quad", eval_type='test')
            val_f1 = val_res.get('micro-F1', 0.0)
            # tp/fp/fn tersedia karena patch_eval_metrics_counts() di sel 5a.
            val_tp = float(val_res.get('tp', float('nan')))
            val_fp = float(val_res.get('fp', float('nan')))
            val_fn = float(val_res.get('fn', float('nan')))

            peak_vram = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if torch.cuda.is_available() else 0.0
            st.step(f"Epoch {epoch:02d} | loss {avg_loss:.4f} | TP {val_tp:.0f} FP {val_fp:.0f} "
                    f"FN {val_fn:.0f} | P {val_res.get('precision', 0.0) * 100:.2f}% "
                    f"| R {val_res.get('recall', 0.0) * 100:.2f}% "
                    f"| F1 {val_f1 * 100:.2f}% | peak VRAM {peak_vram:.0f} MB")

            step1_history.append({
                "epoch": epoch, "loss": avg_loss,
                "tp": val_tp, "fp": val_fp, "fn": val_fn,
                "precision": val_res.get('precision', 0.0),
                "recall": val_res.get('recall', 0.0),
                "micro-F1": val_f1,
                "peak_vram_mb": round(peak_vram, 2)
            })

            if val_f1 > best_step1_f1:
                best_step1_f1 = val_f1
                best1_epoch = epoch
                epochs_since_best_1 = 0  # Reset counter early stopping
                torch.save(model_step1.state_dict(), step1_bin)
                model_step1.config.to_json_file(os.path.join(step1_ckpt, "config.json"))
                tokenizer.save_vocabulary(step1_ckpt)
                st.note(f"🔥 Checkpoint terbaik diperbarui → {step1_ckpt}")
            else:
                epochs_since_best_1 += 1  # Increment counter early stopping

            # ── Rolling epoch checkpoint (resume per-epoch) ───────────────────
            _epoch_ckpt_dir = os.path.join(
                session_dirs["checkpoints"], f"step1_epoch_{epoch}")
            os.makedirs(_epoch_ckpt_dir, exist_ok=True)
            torch.save(model_step1.state_dict(),
                       os.path.join(_epoch_ckpt_dir, "pytorch_model.bin"))
            torch.save(optimizer_1.state_dict(),
                       os.path.join(_epoch_ckpt_dir, "optimizer.pt"))
            st.note(f"💾 Rolling checkpoint epoch {epoch} disimpan")

            # Hapus rolling checkpoint epoch sebelumnya (hemat storage)
            _prev_epoch_dir = os.path.join(
                session_dirs["checkpoints"], f"step1_epoch_{epoch - 1}")
            if epoch > start_epoch and os.path.isdir(_prev_epoch_dir):
                shutil.rmtree(_prev_epoch_dir, ignore_errors=True)
                st.note(f"🗑️  Rolling checkpoint epoch {epoch - 1} dihapus")

            # ── Early stopping check ──────────────────────────────────────────
            if (PATIENCE > 0 and epoch >= MIN_EPOCHS_BEFORE_STOP and 
                epochs_since_best_1 >= PATIENCE):
                st.note(f"⏹️ Early stopping: F1 tidak membaik selama {PATIENCE} epoch "
                        f"(terbaik di epoch {best1_epoch}, F1 {best_step1_f1 * 100:.2f}%)")
                early_stopped_step1 = True
                break
            # ─────────────────────────────────────────────────────────────────

            # Update resume JSON setiap akhir epoch
            with open(step1_resume_json, "w", encoding="utf-8") as _rjfw:
                json.dump({
                    "last_completed_epoch": epoch,
                    "total_epochs": NUM_EPOCHS,
                    "best_micro_f1": best_step1_f1,
                    "best_epoch": best1_epoch,
                    "history": step1_history,
                    "early_stopped": early_stopped_step1,
                    "stopped_at_epoch": epoch if early_stopped_step1 else None,
                    "saved_at": datetime.now().isoformat(),
                }, _rjfw, indent=2)
            # ─────────────────────────────────────────────────────────────────

            # Jejak progres yang bertahan meski runtime terputus di tengah training.
            pd.DataFrame(step1_history).to_csv(step1_csv, index=False, encoding="utf-8")
            write_stage_progress(step1_progress_json, stage="STEP1_TRAINING",
                                 epoch=epoch, total_epochs=NUM_EPOCHS,
                                 last_loss=avg_loss, last_tp=val_tp, last_fp=val_fp,
                                 last_fn=val_fn, last_micro_f1=val_f1,
                                 best_micro_f1=best_step1_f1,
                                 best_epoch=best1_epoch,
                                 peak_vram_mb=round(peak_vram, 2))
            update_mcp_manifest("STEP1_TRAINING", 3, {
                "step1_epoch_progress": f"{epoch}/{NUM_EPOCHS}",
                "step1_best_micro_f1": float(best_step1_f1 * 100),
                "step1_best_epoch": best1_epoch,
            })
            epoch_bar.set_postfix(best_f1=f"{best_step1_f1 * 100:.2f}%", loss=f"{avg_loss:.4f}")
        epoch_bar.close()

        # Hapus rolling checkpoint setelah training selesai penuh
        _last_rolling_dir = os.path.join(
            session_dirs["checkpoints"], f"step1_epoch_{NUM_EPOCHS}")
        if os.path.isdir(_last_rolling_dir):
            shutil.rmtree(_last_rolling_dir, ignore_errors=True)
            st.note(f"🗑️  Rolling checkpoint epoch final dihapus (training selesai penuh)")

        if early_stopped_step1:
            print(f"⏹️ Training berhenti (early stopping) di epoch {epoch}. "
                  f"Micro-F1 terbaik {best_step1_f1 * 100:.2f}% pada epoch {best1_epoch}.", 
                  flush=True)
        else:
            print(f"🏁 Training selesai. Micro-F1 terbaik {best_step1_f1 * 100:.2f}% "
                  f"pada epoch {best1_epoch}.", flush=True)

# Ringkasan satu berkas untuk kedua cabang (training maupun cache hit).
step1_run_json = os.path.join(session_dirs["logs"], "step1_run_result.json")
_best_row1, _best_f1_1, _best_ep1 = best_epoch_row(globals().get("step1_history", []))
with open(step1_run_json, "w", encoding="utf-8") as _jf:
    json.dump({
        "mode": "cache_hit" if STEP1_SKIP_TRAINING else "trained",
        "domain": DOMAIN,
        "total_epochs_target": NUM_EPOCHS,
        "epochs_recorded": len(globals().get("step1_history", [])),
        "best_epoch": _best_ep1 or best1_epoch,
        "best_micro_f1": _best_f1_1,
        "early_stopped": globals().get("early_stopped_step1", False),
        "stopped_at_epoch": epoch if globals().get("early_stopped_step1", False) else None,
        "best_micro_f1_pct": round(_best_f1_1 * 100, 2),
        "best_row": _best_row1,
        "history": globals().get("step1_history", []),
        "checkpoint": step1_ckpt,
        "csv": step1_csv,
        "saved_at": datetime.now().isoformat(),
    }, _jf, indent=2)
print(f"🧾 Ringkasan run Step 1 (termasuk TP/FP/FN per epoch) → {step1_run_json}")

### 5f. Plot, Tabel & Penyimpanan State Step 1
Sel pelaporan: aman dijalankan ulang, baik setelah training maupun setelah cache hit.

In [ ]:
require_vars("step_stage", "step1_history", "best_step1_f1", "best1_epoch")

with step_stage("5f. Plot, tabel laporan, manifest & state Step 1", 5) as st:
    _p1 = os.path.join(plots_dir, "03_step1_training_loss_f1_curve.png")
    if step1_history:
        plot_training_history(
            step1_history, task_name="Step 1 (BERT-CRF)",
            output_plot_path=_p1,
            output_csv_path=step1_csv
        )
        st.step(f"Plot & CSV riwayat ditulis ({len(step1_history)} epoch)")
    else:
        st.step("Riwayat kosong — plot dilewati")

    rep.section("3. Step 1: ekstraksi aspect & opinion")
    df_s1_tabel = history_display_frame(step1_history)
    if not df_s1_tabel.empty:
        _kolom_hitung = [c for c in ("TP", "FP", "FN") if c in df_s1_tabel.columns]
        export_step_table(df_s1_tabel, name="master_03_step1_riwayat", csv_dir=csv_dir,
                          md_dir=md_dir,
                          title=f"Riwayat Training Step 1 ({DOMAIN.upper()})",
                          notes=("Metrik dihitung pada test set tiap epoch. "
                                 + ("TP/FP/FN adalah hitungan mentah span aspect/opinion; "
                                    "Precision/Recall/Micro-F1 dalam persen."
                                    if _kolom_hitung else
                                    "Kolom TP/FP/FN tidak ada — sel 5a belum dijalankan "
                                    "pada run yang menghasilkan riwayat ini.")),
                          max_rows_md=NUM_EPOCHS)
        rep.table(df_s1_tabel, max_rows=NUM_EPOCHS, caption="Metrik step 1 per epoch")
        st.step(f"Tabel master_03_step1_riwayat diekspor ({len(df_s1_tabel)} baris, "
                f"kolom hitungan: {', '.join(_kolom_hitung) or 'tidak ada'})")

        _row1, _f1_1, _ep1 = best_epoch_row(step1_history)
        rep.kv({
            "epoch_terbaik": _ep1 or best1_epoch,
            "micro-F1_terbaik": f"{_f1_1 * 100:.2f}%",
            "TP_FP_FN_epoch_terbaik": (
                f"{_row1.get('tp', float('nan')):.0f} / {_row1.get('fp', float('nan')):.0f} / "
                f"{_row1.get('fn', float('nan')):.0f}" if "tp" in _row1 else "tidak tercatat"),
            "checkpoint": step1_ckpt,
        })
        st.step(f"Micro-F1 terbaik {_f1_1 * 100:.2f}% (epoch {_ep1 or best1_epoch})")
    else:
        st.step("Tabel riwayat dilewati (tidak ada metrik per epoch)")

    if os.path.exists(_p1):
        from IPython.display import Image, display
        display(Image(_p1))
        rep.image(_p1, "Kurva training step 1")

    update_mcp_manifest("STEP1_COMPLETED", 3, {
        "step1_best_micro_f1": float(best_step1_f1 * 100 if best_step1_f1 <= 1.0 else best_step1_f1),
        "step1_checkpoint": step1_ckpt
    })
    st.step("session_manifest.json → STEP1_COMPLETED")

    save_pipeline_state({"best_step1_f1": best_step1_f1, "best_step1_epoch": best1_epoch})
    st.step(f"pipeline_state.pkl diperbarui | pred4pipeline.txt: "
            f"{'ada' if os.path.exists(pred_file) else 'BELUM ADA'}")

## 6. Smart State Checkpoint Saver (`pipeline_state.pkl`)

In [ ]:
# Simpan status variabel pipeline ke file pickle untuk pemulihan cepat.
# Menyimpan parameter konfigurasi, direktori, model status, dan artefak runtime.
checkpoint_state_path = save_pipeline_state()

print(f"✅ Pipeline State (expanded) berhasil disimpan ke: {checkpoint_state_path}")
print(f"   Checkpoint Step 1 : {session_dirs['step1_checkpoint']}")
print(f"   Prediksi File     : {os.path.join(session_dirs['logs'], 'pred4pipeline.txt')}")
print("ℹ️ Jika runtime Colab terputus, jalankan sel pemulihan (6b & 6c) di bawah ini untuk melanjutkan langsung ke Step 2.")